# Task 02 — Silver: Orders Clean & Enrich — **SOLUTION**

**Workshop: Final Pipeline | Layer 2 of 3**

## Goal

Read `bronze.lab_orders`, apply transformations, write to `silver.lab_orders`.

```
bronze.lab_orders (all columns are strings!)
        |
        v  Step 1: CAST to correct types
        v  Step 2: Apply transformations
        v
silver.lab_orders
```

## Step 1: Cast to correct data types ⚠️

**IMPORTANT: Bronze table has all columns as STRING type. First, cast to correct types:**

| Column | From | To |
|--------|------|----|
| `order_datetime` | String | TimestampType |
| `quantity` | String | IntegerType |
| `unit_price` | String | DoubleType |
| `discount_percent` | String | DoubleType |
| `total_amount` | String | DoubleType |

## Step 2: Apply transformations

**Then, apply the following transformations:**

| # | What | Details |
|---|------|--------|
| 1 | `withColumn` | Add `net_amount` = `total_amount` × (1 - `discount_percent` / 100), rounded to 2 dp |
| 2 | `when / otherwise` | Add `order_tier`: `"Premium"` >= 500 * `"Standard"` >= 200 * `"Small"` otherwise |
| 3 | `filter` | Remove rows where `order_id` IS NULL |
| 4 | `drop` | Remove `_source_file` (Bronze metadata, not needed in Silver) |

> `net_amount` is used by Task 03 for revenue — make sure the formula is correct.

In [0]:
%run ../../../setup/00_setup

In [0]:
dbutils.widgets.text("catalog",       CATALOG,       "Catalog")
dbutils.widgets.text("bronze_schema", BRONZE_SCHEMA, "Bronze Schema")
dbutils.widgets.text("silver_schema", SILVER_SCHEMA, "Silver Schema")

catalog       = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

source_table = f"{catalog}.{bronze_schema}.lab_orders"
target_table = f"{catalog}.{silver_schema}.lab_orders"

print(f"Source : {source_table}")
print(f"Target : {target_table}")

## Exercise 1 — Imports

Import all PySpark functions needed for:
* Casting columns to correct types
* Applying transformations

In [0]:
from pyspark.sql.functions import col, to_timestamp, when, round, lit

---

## Exercise 2 — Read the Bronze table

In [0]:
# Read the Bronze table into orders_bronze (Task 01 must have run first)
orders_bronze = spark.table(source_table)

In [0]:
print(f"Bronze rows : {orders_bronze.count():,}")
orders_bronze.printSchema()

## Exercise 3 — Cast types FIRST, then apply transformations

⚠️ **Remember**: Bronze has all columns as STRING. 

Chain all operations into a single expression:
1. **First**: Cast all numeric columns to correct types (see Step 1 above)
2. **Then**: Apply transformations (see Step 2 above)

Name the result `orders_silver`.

In [0]:
orders_silver = (
    orders_bronze
    # ===== STEP 1: Cast to correct data types =====
    .withColumn("order_datetime", to_timestamp(col("order_datetime"), "yyyy-MM-dd'T'HH:mm:ss"))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("unit_price", col("unit_price").cast("double"))
    .withColumn("discount_percent", col("discount_percent").cast("double"))
    .withColumn("total_amount", col("total_amount").cast("double"))
    
    # ===== STEP 2: Apply transformations =====
    # 1. Add net_amount = total_amount × (1 - discount_percent / 100), rounded to 2 dp
    .withColumn("net_amount",
        round(col("total_amount") * (lit(1) - col("discount_percent") / lit(100)), 2))
    # 2. Add order_tier based on net_amount
    .withColumn("order_tier",
        when(col("net_amount") >= 500, "Premium")
        .when(col("net_amount") >= 200, "Standard")
        .otherwise("Small"))
    # 3. Remove rows where order_id is null
    .filter(col("order_id").isNotNull())
    # 4. Drop Bronze metadata column not needed in Silver
    .drop("_source_file")
)

In [0]:
print(f"Silver rows : {orders_silver.count():,}")
orders_silver.select(
    "order_id","order_datetime","total_amount",
    "discount_percent","net_amount","order_tier"
).display(10, truncate=False)
orders_silver.printSchema()

---

## Exercise 4 — Write to Silver as a managed Delta table

In [0]:
(
    orders_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
)

In [0]:
row_count = spark.table(target_table).count()
print(f"OK  {target_table}  ->  {row_count:,} rows")
display(spark.table(target_table).limit(5))

In [0]:
import json
dbutils.notebook.exit(json.dumps({"status":"SUCCESS","table":target_table,"rows":row_count}))